# 1. Data stage (CPU session, no GPU quota used)

Kaggle settings: **Accelerator: None**, **Internet: On**. Add your Hugging Face token under
**Add-ons -> Secrets** as `HF_TOKEN` (only English/Svarah needs it).
Run with **Save Version -> Save & Run All (Commit)**. Output streams live to the log and
any failing step stops the run.

In [ ]:
import os, subprocess, glob, shutil, json, sys, time
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
subprocess.run(["git", "pull"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.environ["TQDM_MININTERVAL"] = "60"   # keep progress bars from flooding the log
os.environ["PYTHONUNBUFFERED"] = "1"


def sh(cmd):
    """Run a bash command, stream its output live, raise if it fails."""
    print("$", cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -euo pipefail; " + cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit {}): {}".format(p.returncode, cmd))
    print("[ok in {:.1f} min]".format((time.time() - t0) / 60), flush=True)

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (needed for English):", e)

In [ ]:
sh("pip install -q -r requirements.txt")
sh("bash scripts/setup_mfa.sh")

In [ ]:
sh("bash scripts/run_pipeline.sh data telugu")
s = json.load(open("data/Telugu/preprocessed/summary.json"))
print(json.dumps(s, indent=1)); print(open("Results/Telugu/others/alignment_coverage.txt").read())
assert s["train_items"] > 1000, "too few Telugu training items"

In [ ]:
sh("bash scripts/run_pipeline.sh data english")
s = json.load(open("data/English/preprocessed/summary.json"))
print(json.dumps(s, indent=1)); print(open("Results/English/others/alignment_coverage.txt").read())
assert s["train_items"] > 1000, "too few English training items"

In [ ]:
# both languages have features: drop what training does not need (MFA env, raw audio)
sh("rm -rf .mfa_env .micromamba mfa_models data/*/raw/*/ data/*/TextGrid && du -sh data/*/preprocessed")